In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Question 1

In [2]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Read the image at reduced resolution
im = cv.imread('images/the_berry_farms_sunflower_field.jpeg', cv.IMREAD_REDUCED_COLOR_4)
gray = cv.cvtColor(im, cv.COLOR_BGR2GRAY)
gray = gray.astype(np.float32) / 255.0  # normalize


In [4]:
# Parameters
sigma_min = 0.1       # starting sigma
sigma_max = 30      # max sigma (depends on expected sunflower size)
num_scales = 50
k = (sigma_max / sigma_min) ** (1.0 / (num_scales - 1))

# List to store LoG responses
h, w = gray.shape
scale_space = np.zeros((h, w, num_scales), dtype=np.float32)
sigmas = []

sigma = sigma_min
for i in range(num_scales):
    sigmas.append(sigma)
    # Apply Gaussian blur first
    blurred = cv.GaussianBlur(gray, (0, 0), sigmaX=sigma, sigmaY=sigma)
    # Compute Laplacian
    log = cv.Laplacian(blurred, cv.CV_32F)
    # Normalize by sigma^2 (important for scale-normalized LoG)
    scale_space[:, :, i] = (sigma**2) * np.abs(log)
    sigma *= k


In [5]:
from scipy.ndimage import maximum_filter

# Detect 3D local maxima
coordinates = []
threshold = 0.03  # threshold for blob response, adjust if needed

for i in range(num_scales):
    max_img = maximum_filter(scale_space[:, :, i], size=3)
    mask = (scale_space[:, :, i] == max_img) & (scale_space[:, :, i] > threshold)
    y, x = np.nonzero(mask)
    for xi, yi in zip(x, y):
        coordinates.append((xi, yi, sigmas[i]))


In [6]:
# Compute radii
blobs = [(x, y, np.sqrt(2)*s) for (x, y, s) in coordinates]

# Sort by radius to find largest
blobs_sorted = sorted(blobs, key=lambda b: b[2], reverse=True)
largest_blobs = blobs_sorted[:5]  # report top 5 largest circles


In [ ]:
output = im.copy()
for (x, y, r) in largest_blobs:
    cv.circle(output, (int(x), int(y)), int(r), (0, 255, 0), 2)

plt.figure(figsize=(12, 8))
plt.imshow(cv.cvtColor(output, cv.COLOR_BGR2RGB))
plt.title('Detected Sunflowers (Largest Circles)')
plt.axis('off')
plt.show()


NameError: name 'sunflower_img' is not defined

In [62]:
print("σ range used:", sigma_min, "to", sigma_max)
print("Largest circles (x, y, radius):")
for (x, y, r) in largest_blobs:
    print(f"x={x}, y={y}, radius={r:.2f}")


σ range used: 1 to 30
Largest circles (x, y, radius):
x=1, y=0, radius=42.43
x=66, y=0, radius=42.43
x=69, y=0, radius=42.43
x=110, y=0, radius=42.43
x=157, y=0, radius=42.43
